## CREATING A CUSTOMER SUPPORT AUTOMATION WITH CREWAI :

In [1]:
from crewai import Agent, Task, Crew, LLM

In [2]:
# WARNINGS IGNORING:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv()
print("API Key loaded:", bool(os.getenv("GEMINI_API_KEY")))

API Key loaded: True


In [ ]:
import os
os.environ["GEMINI_API_KEY"] = "YOUR-GEMINI-API-KEY-HERE"
os.environ["GOOGLE_API_KEY"] = "YOUR-GEMINI-API-KEY-HERE"

print("Key set:", os.getenv("GEMINI_API_KEY")[:5])  # prints first 10 chars to confirm

Key set: AIzaS


In [5]:
gemini_llm = LLM(
    model="gemini/gemini-3.1-pro-preview",
    temperature=0.0
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


#### CREATING AGENTS FOR CUSTOMER SUPPORT :
Here, we will create agents for the Customer Support for the company called `Stripe`, which is a SaaS Fintech company.

##### Role Playing, Focus and Cooperation between Agents :

In [6]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at Stripe (https://stripe.com/)"
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	llm=gemini_llm,
	allow_delegation=False,
	verbose=True
)

We set `allow_delegation=False` because if its not set into False, the agent will delegate the task to the other one while its just assigned its role and focus. We are avoiding that possibility.

In [7]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at Stripe (https://stripe.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	llm=gemini_llm,
	verbose=True
)

We have set until now the following : 

* **Role Playing**: Both agents have been given a role, goal and backstory.
* **Focus**: Both agents have been prompted to get into the character of the roles they are playing.
* **Cooperation**: Support Quality Assurance Agent can delegate work back to the Support Agent, allowing for these agents to work together.

### TOOLS, MEMORY AND GUARDRAILS IN CREWAI :

##### TOOLS :

In [8]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

#### Here, 
SerperDevTool -> search tool( ) \
ScrapeWebsitetool -> scrape tool( )

#### Initializing the Docs scraper tool :

In [9]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.stripe.com/sdks/server-side"
)

#### CREATING TASKS :

In [10]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

Here,
- `quality_assurance_review` is not using any Tool(s)
- Here the QA Agent will only review the work of the Support Agent"

In [11]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)

#### CREATING THE CREW :

Memory
- Setting `memory=True` when putting the crew together enables Memory.

In [12]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True,
  memory=False,
  llm=gemini_llm
)

#### GUARDRAILS :

In [ ]:
inputs = {
    "customer": "TechFlow India",
    "person": "Arjun Mehta",
    "inquiry": (
        "I need help with migrating to the new StripeClient in Python, "
        "specifically I am currently using the old global configuration "
        "pattern like stripe.api_key = 'stripe-api-key-here' and calling "
        "stripe.PaymentIntent.create(). "
        "How should I update my code to use the new StripeClient pattern? "
        "Can you provide guidance?"
    )
}
result = crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 441affd0-ba85-41d3-b631-f696b2469322                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: TechFlow India just reached out with a super important ask:                                              │
│  I need help with migrating to the new StripeClient in Python, specifically I am currently using the old        │
│  global configuration pattern like stripe.api_key = 'stripe-api-key-here' and calling                           │
│  stripe.PaymentIntent.create(). How should I update my code to use the new StripeClient pattern? Can you        │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Arjun Mehta from TechFlow India is the one that reached out. Make sure to use everything you know to provide   │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│  ID: e2cc0b82-7198-40c1-88a2-770e9937ffac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: TechFlow India just reached out with a super important ask:                                              │
│  I need help with migrating to the new StripeClient in Python, specifically I am currently using the old        │
│  global configuration pattern like stripe.api_key = 'stripe-api-key-here' and calling                           │
│  stripe.PaymentIntent.create(). How should I update my code to use the new StripeClient pattern? Can you        │
│  provide guidance?                                                                                              │
│                                                                                                                 │
│  Arjun Mehta from TechFlow India is the one that reached out. Make sure to use everything you know to provide   │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_website_content                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_website_content executed with result: The following text is scraped website content:
Introduction to server-side SDKs | Stripe Documentation
Skip to content Server-side SDKs Create account or Sign in The Stripe Docs logo Search / Ask AI C...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Task: Review the response drafted by the Senior Support Representative for TechFlow India's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hey Arjun,                                                                                                     │
│                                                                                                                 │
│  Thanks for reaching out to Stripe Support! We love seeing what the team at TechFlow India is building, and     │
│  I’d be more than happy to help you with your migration to the new `StripeClient` pattern in Python.            │
│                                                                                                                 │
│  Migrating to the new `StripeClient` is a fantastic move for your codebase. As outlined in our [Server-side     │
│  SDKs documentation](https://docs.stripe.com/sdks/server-side), the `StripeClient` class was introduced in      │
│  version **8.0.0** of the `stripe-python` SDK. It acts as a dedicated, thread-safe entry point to help you      │
│  make requests to the Stripe API without relying on global state variables.                                     │
│                                                                                                                 │
│  Here is a complete guide on how to update your code, along with a few pro-tips to make the transition          │
│  seamless.                                                                                                      │
│                                                                                                                 │
│  ### 1. The Old Pattern (Global Configuration)                                                                  │
│  Previously, you would set the API key globally and call static methods directly on the resource classes.       │
│                                                                                                                 │
│  **Old Code Example:**                                                                                          │
│  ```python                                                                                                      │
│  import stripe                                                                                                  │
│                                                                                                                 │
│  # Global configuration                                                                                         │
│  stripe.api_key = 'sk_test_your_api_key_here'                                                                   │
│                                                                                                                 │
│  # Creating a PaymentIntent using static methods                                                                │
│  payment_intent = stripe.PaymentIntent.create(                                                                  │
│      amount=2000,                                                                                               │
│      currency='usd',                                                                                            │
│      payment_method_types=['card']                                                                              │
│  )                                                                                                              │
│  print(payment_intent.id)                              

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for TechFlow India's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 441affd0-ba85-41d3-b631-f696b2469322                                                                       │
│  Final Output: Hey Arjun,                                                                                       │
│                                                                                                                 │
│  Thanks for reaching out to Stripe Support! We love seeing what the team at TechFlow India is building, and     │
│  I’d be more than happy to help you with your migration to the new `StripeClient` pattern in Python.            │
│                                                                                                                 │
│  Migrating to the new `StripeClient` is a fantastic move for your codebase. As outlined in our [Server-side     │
│  SDKs documentation](https://docs.stripe.com/sdks/server-side), the `StripeClient` class was introduced in      │
│  version **8.0.0** of the `stripe-python` SDK. It acts as a dedicated, thread-safe entry point to help you      │
│  make requests to the Stripe API without relying on global state variables.                                     │
│                                                                                                                 │
│  Here is a complete guide on how to update your code, along with a few pro-tips to make the transition          │
│  seamless.                                                                                                      │
│                                                                                                                 │
│  ### 1. The Old Pattern (Global Configuration)                                                                  │
│  Previously, you would set the API key globally and call static methods directly on the resource classes.       │
│                                                                                                                 │
│  **Old Code Example:**                                                                                          │
│  ```python                                                                                                      │
│  import stripe                                                                                                  │
│                                                                                                                 │
│  # Global configuration                                                                                         │
│  stripe.api_key = 'sk_test_your_api_key_here'                                                                   │
│                                                                                                                 │
│  # Creating a PaymentIntent using static methods                                                                │
│  payment_intent = stripe.PaymentIntent.create(                                                                  │
│      amount=2000,                                                                                               │
│      currency='usd',                                                                                            │
│      payment_method_types=['card']                                                                              │
│  )                                                                                                              │
│  print(payment_intent.id)                             

### Finally, displaying the result as a Markdown

In [14]:
from IPython.display import Markdown
display(Markdown(result.raw))


Hey Arjun,

Thanks for reaching out to Stripe Support! We love seeing what the team at TechFlow India is building, and I’d be more than happy to help you with your migration to the new `StripeClient` pattern in Python. 

Migrating to the new `StripeClient` is a fantastic move for your codebase. As outlined in our [Server-side SDKs documentation](https://docs.stripe.com/sdks/server-side), the `StripeClient` class was introduced in version **8.0.0** of the `stripe-python` SDK. It acts as a dedicated, thread-safe entry point to help you make requests to the Stripe API without relying on global state variables.

Here is a complete guide on how to update your code, along with a few pro-tips to make the transition seamless.

### 1. The Old Pattern (Global Configuration)
Previously, you would set the API key globally and call static methods directly on the resource classes. 

**Old Code Example:**
```python
import stripe

# Global configuration
stripe.api_key = 'sk_test_your_api_key_here'

# Creating a PaymentIntent using static methods
payment_intent = stripe.PaymentIntent.create(
    amount=2000,
    currency='usd',
    payment_method_types=['card']
)
print(payment_intent.id)
```

### 2. The New Pattern (StripeClient)
With the new pattern, you instantiate a `StripeClient` object with your API key. All API resources are then accessed as properties on this client instance (e.g., `client.payment_intents`, `client.customers`, `client.checkout.sessions`). 

*Note: While you can pass a `params` dictionary, the most Pythonic way (and our recommended approach) is to pass your parameters directly as keyword arguments.*

**New Code Example:**
```python
import stripe

# 1. Initialize the StripeClient with your API key
# Pro-tip: You can also pin your Stripe API version right here!
client = stripe.StripeClient('sk_test_your_api_key_here')

# 2. Access the resource via the client instance using keyword arguments
payment_intent = client.payment_intents.create(
    amount=2000,
    currency="usd",
    payment_method_types=['card']
)

print(payment_intent.id)
```

### 💡 A Quick Note on Webhooks
One common question we get during this migration is about webhooks. Please note that webhook signature verification doesn't require the client instance. You will continue to use the static utility method `stripe.Webhook.construct_event(payload, sig_header, endpoint_secret)` just as you did before!

### Why make this switch?
Migrating to the `StripeClient` pattern provides several awesome benefits for TechFlow India's architecture:
* **Multiple Configurations:** You can simultaneously use multiple clients with different configuration options (like different API keys for different regions, or specific `Stripe-Account` headers for Connect) without them interfering with each other.
* **Thread Safety:** Removing global state mutations (`stripe.api_key = ...`) makes your application much safer and more predictable in multi-threaded environments.
* **Easier Testing & Mocking:** Because `StripeClient` doesn't use static methods or global state, it is significantly easier to mock during your unit testing.
* **Fewer API Calls:** In older SDK patterns, you sometimes had to call a `retrieve` before doing an `update` or `delete`. With `StripeClient`, you can access all API endpoints directly with a single method call.

### References & Further Reading
* **Stripe Server-Side SDKs Overview:** [https://docs.stripe.com/sdks/server-side](https://docs.stripe.com/sdks/server-side)
* **Stripe-Python GitHub Repository & v8 Migration Guide:** [https://github.com/stripe/stripe-python](https://github.com/stripe/stripe-python) 

I hope this fully answers your question and makes the migration process smooth for your team! If you or anyone else at TechFlow India have any more questions, need help mapping out specific endpoints, or run into any roadblocks, just drop a reply to this message. I'm always here and happy to help.

Best regards,

**Senior Support Representative**
Stripe Support

### Lets bring RAG into this Customer Support Automation :

In [15]:
%pip install google-generativeai

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
os.environ["OPENAI_API_KEY"] = "fake-key"

from crewai_tools import WebsiteSearchTool

# Give it the ROOT domain — it searches all pages under it
# We configure the tool to use Gemini's LLM and Embeddings instead of OpenAI
# Forcing the gemini key directly into the config dictionaries

gemini_api_key = os.environ.get("GEMINI_API_KEY")

gemini_config = {
    "embedding_model": {
        "provider": "google-generativeai",
        "config": {
            "model": "models/embedding-001",
            "task_type": "retrieval_document",
        "api_key" : gemini_api_key,
        }
    }
}
stripe_search_tool = WebsiteSearchTool(
    website="https://docs.stripe.com",
    config=gemini_config,

collection_name="stripe_gemini_docs_clean"
)

print("Successfully injected Gemini API key!!")

Successfully injected Gemini API key!!


#### Including the RAG tool here :

In [17]:
# 1. Attach our working RAG tool to the existing Support Agent
support_agent.tools = [stripe_search_tool]

# 2. Define the wide range of questions we want to test
customer_questions = [
    "How do I set up subscriptions?",
    "How do I handle refunds?",
    "How do I go live?"
]

# 3. Loop through and let the Crew process each one dynamically!
for question in customer_questions:
    print(f"\n==============================================")
    print(f"🕵️‍♂️ PROCESSING NEW CUSTOMER INQUIRY: '{question}'")
    print(f"==============================================\n")
    
    # Update the inputs dictionary for this specific run
    inputs = {
        "customer": "TechFlow India",
        "person": "Arjun Mehta",
        "inquiry": question
    }
    
    # Kickoff the exact same crew!
    result = crew.kickoff(inputs=inputs)
    
    # Display the result formatted nicely
    from IPython.display import Markdown, display
    display(Markdown(result.raw))



🕵️‍♂️ PROCESSING NEW CUSTOMER INQUIRY: 'How do I set up subscriptions?'



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 441affd0-ba85-41d3-b631-f696b2469322                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: TechFlow India just reached out with a super important ask:                                              │
│  How do I set up subscriptions?                                                                                 │
│                                                                                                                 │
│  Arjun Mehta from TechFlow India is the one that reached out. Make sure to use everything you know to provide   │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│  ID: e2cc0b82-7198-40c1-88a2-770e9937ffac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: TechFlow India just reached out with a super important ask:                                              │
│  How do I set up subscriptions?                                                                                 │
│                                                                                                                 │
│  Arjun Mehta from TechFlow India is the one that reached out. Make sure to use everything you know to provide   │
│  the best support possible.You must strive to provide a complete and accurate response to the customer's        │
│  inquiry.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hello again, Arjun!                                                                                            │
│                                                                                                                 │
│  It is absolutely wonderful to hear from you and the TechFlow India team again! I would be more than happy to   │
│  help you set up subscriptions. Since we just worked on migrating your Python code to the new `StripeClient`    │
│  pattern, I will make sure to use that exact pattern in the examples below so you can drop this right into      │
│  your updated codebase!                                                                                         │
│                                                                                                                 │
│  Setting up subscriptions is handled by **Stripe Billing**. To get a subscription up and running, you           │
│  generally need to follow a 4-step process: creating a Product and Price, creating a Customer, collecting       │
│  payment details, and finally, provisioning the subscription.                                                   │
│                                                                                                                 │
│  Here is your comprehensive, step-by-step guide to setting this up:                                             │
│                                                                                                                 │
│  ### Step 1: Create a Product and a Price                                                                       │
│  Before you can subscribe a customer, you need to define what they are subscribing to (the **Product**) and     │
│  how much it costs / how often they are billed (the **Price**). You can do this manually in the Stripe          │
│  Dashboard, or programmatically via the API.                                                                    │
│                                                                                                                 │
│  Here is how you do it via the API using your new `StripeClient`:                                               │
│                                                                                                                 │
│  ```python                                                                                                      │
│  import stripe                                                                                                  │
│                                                                                                                 │
│  client = stripe.StripeClient('sk_test_your_api_key_here')                                                      │
│                                                                                                                 │
│  # 1. Create the Product                                                                                        │
│  product = client.products.create(                                                                              │
│      name="TechFlow Pro Plan",                                                                                  │
│      description="Monthly access to TechFlow Pro features"                                                      │
│  )                                                     

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Hey Arjun!                                                                                                     │
│                                                                                                                 │
│  Congratulations on reaching this massive milestone! Going live is an incredibly exciting step for TechFlow     │
│  India, and I'm absolutely thrilled to help you cross the finish line. You've done a fantastic job migrating    │
│  to the new Python `StripeClient`, setting up your subscriptions, and preparing your refund logic. Now, it's    │
│  time to turn it all on for real customers!                                                                     │
│                                                                                                                 │
│  Transitioning from Test Mode to Live Mode involves a few critical steps to ensure your business is fully       │
│  verified, your code is pointing to the right environment, and your data is ready.                              │
│                                                                                                                 │
│  Here's your comprehensive, step-by-step guide to going live with Stripe:                                       │
│                                                                                                                 │
│  ### Step 1: Activate Your Stripe Account                                                                       │
│  Before you can process real money, Stripe needs to verify your business details to comply with financial       │
│  regulations.                                                                                                   │
│  1. Log in to your Stripe Dashboard.                                                                            │
│  2. In the top left corner or via the banner at the top, click on **Activate payments**.                        │
│  3. Fill out the required business information. Since TechFlow India is based in India, you'll need to provide  │
│  specific KYC (Know Your Customer) details, such as your Company PAN, GSTIN (if applicable), business address,  │
│  and the personal details of the business owners/executives.                                                    │
│  4. Add your **Bank Account details** so Stripe knows exactly where to send your payouts!                       │
│                                                                                                                 │
│  ### Step 2: Swap Your API Keys                                                                                 │
│  Right now, your application is using test API keys (which look like `pk_test_...` and `sk_test_...`). To       │
│  process real transactions, you must swap these out for your live API keys (`pk_live_...` and `sk_live_...`).   │
│                                                                                                                 │
│  **Frontend (Stripe.js / Elements / Checkout):**                                                                │
│  Update your publishable key to your live key: `pk_live_your_live_key_here`.                                    │
│                                                                                                                 │
│  **Backend (Python `StripeClient`):**                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for TechFlow India's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Hey Arjun!

Congratulations on reaching this massive milestone! Going live is an incredibly exciting step for TechFlow India, and I'm absolutely thrilled to help you cross the finish line. You've done a fantastic job migrating to the new Python `StripeClient`, setting up your subscriptions, and preparing your refund logic. Now, it's time to turn it all on for real customers!

Transitioning from Test Mode to Live Mode involves a few critical steps to ensure your business is fully verified, your code is pointing to the right environment, and your data is ready. 

Here's your comprehensive, step-by-step guide to going live with Stripe:

### Step 1: Activate Your Stripe Account
Before you can process real money, Stripe needs to verify your business details to comply with financial regulations.
1. Log in to your Stripe Dashboard.
2. In the top left corner or via the banner at the top, click on **Activate payments**.
3. Fill out the required business information. Since TechFlow India is based in India, you'll need to provide specific KYC (Know Your Customer) details, such as your Company PAN, GSTIN (if applicable), business address, and the personal details of the business owners/executives.
4. Add your **Bank Account details** so Stripe knows exactly where to send your payouts!

### Step 2: Swap Your API Keys
Right now, your application is using test API keys (which look like `pk_test_...` and `sk_test_...`). To process real transactions, you must swap these out for your live API keys (`pk_live_...` and `sk_live_...`).

**Frontend (Stripe.js / Elements / Checkout):**
Update your publishable key to your live key: `pk_live_your_live_key_here`.

**Backend (Python `StripeClient`):**
Update the initialization of your client to use your live secret key. *Pro-tip: Always store this in an environment variable rather than hardcoding it in your codebase!*

```python
import stripe
import os

# Fetch the live key securely from your environment variables
live_api_key = os.environ.get('STRIPE_SECRET_KEY') 
client = stripe.StripeClient(live_api_key) # Starts with sk_live_...
```

### Step 3: Recreate Your Products and Prices
Data created in Test Mode (like the "TechFlow Pro Plan" Product and Price we set up earlier) **does not** automatically transfer to Live Mode. 
* You'll need to recreate your Products and Prices in the Live Dashboard (or programmatically via your live API keys).
* **Important:** When you recreate them, they will generate *new* IDs (e.g., a new `price_...` ID). Make sure to update your code or database with these new Live Price IDs so your Checkout Sessions charge the correct amount!

### Step 4: Set Up Live Webhooks
Just like your Products, your Test Mode webhook endpoints don't carry over. 
1. Go to the **Developers > Webhooks** section of your Dashboard.
2. Ensure the "Test mode" toggle in the top right is switched **OFF**.
3. Click **Add an endpoint** and enter your live production URL (e.g., `https://api.techflow.in/webhooks/stripe`).
4. Select the events you want to listen to (like `checkout.session.completed`, `invoice.paid`, `charge.refunded`).
5. **Crucial:** Grab your new Live Webhook Signing Secret (`whsec_...`) and update it in your production environment variables so your server can verify the live events securely.

### Step 5: Perform a Live Test Transaction
Before announcing your launch to the world, we highly recommend running a real transaction yourself to ensure everything is flowing perfectly end-to-end.
1. Go to your live website.
2. Sign up for a TechFlow India subscription using a **real credit or debit card** (test cards like the `4242` card will be declined in live mode).
3. Verify that the payment succeeds, the subscription is created in your Live Stripe Dashboard, and your live webhooks successfully provision the service in your database.
4. Finally, use the refund knowledge we discussed previously to refund your own transaction from the Dashboard or API!

### 🇮🇳 Important Considerations for Indian Businesses
Since TechFlow India is operating in India, please keep these local regulations in mind:
* **e-Mandates for Subscriptions:** The Reserve Bank of India (RBI) requires an Additional Factor of Authentication (AFA) / 3D Secure for the first payment of a subscription, and subsequent recurring payments are handled via e-mandates. The great news is that Stripe Billing and Stripe Checkout automatically handle this complex compliance for you!
* **International Payments:** If you plan to accept payments from outside of India, you'll need to ensure you have submitted your Importer Exporter Code (IEC) or an appropriate Purpose Code in your Stripe Dashboard settings to comply with FEMA guidelines.

### References & Further Reading
To ensure you have all the official documentation at your fingertips, here are the resources I used to build this checklist:
* **The Stripe Go-Live Checklist:** [https://docs.stripe.com/checklist](https://docs.stripe.com/checklist)
* **API Keys Documentation:** [https://docs.stripe.com/keys](https://docs.stripe.com/keys)
* **India-Specific Payment Guidelines:** [https://docs.stripe.com/india](https://docs.stripe.com/india)
* **Testing a Live Integration:** [https://docs.stripe.com/testing#live-testing](https://docs.stripe.com/testing#live-testing)

Arjun, I'm so incredibly proud of the work you and the TechFlow India team have done to get to this point. Please take your time going through these steps, and if you hit any snags whatsoever while flipping the switch to live, just reply to this thread immediately. I'm here to ensure your launch is a massive success!

Wishing you a flawless launch!

Best,

**Senior Support Representative**
Stripe Support

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 441affd0-ba85-41d3-b631-f696b2469322                                                                       │
│  Final Output: Hey Arjun!                                                                                       │
│                                                                                                                 │
│  Congratulations on reaching this massive milestone! Going live is an incredibly exciting step for TechFlow     │
│  India, and I'm absolutely thrilled to help you cross the finish line. You've done a fantastic job migrating    │
│  to the new Python `StripeClient`, setting up your subscriptions, and preparing your refund logic. Now, it's    │
│  time to turn it all on for real customers!                                                                     │
│                                                                                                                 │
│  Transitioning from Test Mode to Live Mode involves a few critical steps to ensure your business is fully       │
│  verified, your code is pointing to the right environment, and your data is ready.                              │
│                                                                                                                 │
│  Here's your comprehensive, step-by-step guide to going live with Stripe:                                       │
│                                                                                                                 │
│  ### Step 1: Activate Your Stripe Account                                                                       │
│  Before you can process real money, Stripe needs to verify your business details to comply with financial       │
│  regulations.                                                                                                   │
│  1. Log in to your Stripe Dashboard.                                                                            │
│  2. In the top left corner or via the banner at the top, click on **Activate payments**.                        │
│  3. Fill out the required business information. Since TechFlow India is based in India, you'll need to provide  │
│  specific KYC (Know Your Customer) details, such as your Company PAN, GSTIN (if applicable), business address,  │
│  and the personal details of the business owners/executives.                                                    │
│  4. Add your **Bank Account details** so Stripe knows exactly where to send your payouts!                       │
│                                                                                                                 │
│  ### Step 2: Swap Your API Keys                                                                                 │
│  Right now, your application is using test API keys (which look like `pk_test_...` and `sk_test_...`). To       │
│  process real transactions, you must swap these out for your live API keys (`pk_live_...` and `sk_live_...`).   │
│                                                                                                                 │
│  **Frontend (Stripe.js / Elements / Checkout):**                                                                │
│  Update your publishable key to your live key: `pk_live_your_live_key_here`.                                    │
│                                                                                                                 │
│  **Backend (Python `StripeClient`):**                 